# Differential expression with donor structure

*Python port of the R vignette [Differential expression with donor structure](https://gregorlueg.github.io/bixverse/articles/differential_expression.html). Original author: Gregor Lueg.*

## Intro

You have 5,355 monocytes from eight donors and you want to know which genes
respond to a treatment. The obvious move is a Wilcoxon test over cells. It is
also the wrong one, and the reason is worth being precise about.

Those 5,355 cells are not 5,355 independent observations. They came from eight
people. Cells from the same donor are correlated, because donors differ in
genotype, in batch, in how the sample was handled, in everything. A test that
treats each cell as independent has an effective sample size of eight and
thinks it has 5,355, so its p-values are far too small. This is
pseudoreplication, and in single cell data it is the default failure mode
rather than an exotic one.

There are two honest ways out.

**Pseudobulk.** Sum the counts per donor per condition and run a bulk method.
The unit of replication becomes the donor, which is correct. You throw away
everything the cell-level data told you about variability within a donor, and
with few donors you end up with very little power.

**A mixed model.** Keep the cells, but model the donor as a random effect.
[NEBULA](https://doi.org/10.1038/s42003-021-02146-6) fits a negative binomial
gamma mixed model per gene, splitting the variance into a subject-level term
and a cell-level term. You keep the cell-level information and still pay the
right price for the donor structure.

`bixverse` gives you all three, and this vignette runs them side by side, first
on a real effect and then on no effect at all. The second one is the
interesting bit.

In [1]:
import tempfile
from pathlib import Path

import numpy as np
import polars as pl

import bixverse as bv
from bixverse.params import EdgerQlParams, MinQualityParams, NebulaParams

## The data

[Kang, et al.](https://doi.org/10.1038/nbt.4042) pooled PBMCs from eight lupus
donors, split each sample in two, and stimulated one half with IFN-β. Every
donor contributes to both arms, so the design is paired and the donor is a
genuine random effect. Roughly 29k cells before quality control.

The data comes as an h5ad: raw counts in `X`, the cell annotations in obs and
the gene annotations in var. `load_h5ad` takes it directly and the counts go
through the usual Rust quality control and normalisation.

In [2]:
kang_path = bv.io.download_kang_pbmc()

tempdir_kang = tempfile.TemporaryDirectory()

sc = bv.io.load_h5ad(
    Path(tempdir_kang.name) / "kang",
    kang_path,
    qc=MinQualityParams(
        min_unique_genes=200,
        min_lib_size=500,
        min_cells=20,
        target_size=1e4,
    ),
    verbose=True,
)

Step 1/3: Analysing data structure and calculating QC metrics...
  Reading CSR matrix structure (shape: 29_065 x 35_635 )...
First pass - gene expression statistics:
  Processed 30% of chunks (1/3)
  Processed 70% of chunks (2/3)
  Processed 100% of chunks (3/3)
  Gene expression stats: min = 0 | max = 28_991 | avg = 493 cells per gene
First pass done: 39.37ms
  Genes passing filter: 12_526 / 35_635
Second pass - cell statistics:
  Processed 30% of chunks (1/3)
  Processed 70% of chunks (2/3)
  Processed 100% of chunks (3/3)
  Cell stats: genes per cell: min = 12 | max = 2_724
  Cell stats: library size: min = 559 | max = 11747
Second pass done: 77.81ms
  Cells passing filter: 28_871 / 29_065
Step 2/3: QC Results:
  Genes passing QC: 12_526 / 35_635
  Cells passing QC: 28_871 / 29_065
Step 3/3: Writing cells to CSR format...
  Processing 28_871 cells in batches of 1_000...
  Processed 1_000 / 28_871 cells
  Processed 3_000 / 28_871 cells
  Processed 5_000 / 28_871 cells
  Processed 7_0

  Processed 15_000 / 28_871 cells
  Processed 17_000 / 28_871 cells
  Processed 19_000 / 28_871 cells
  Processed 21_000 / 28_871 cells
  Processed 23_000 / 28_871 cells
  Processed 25_000 / 28_871 cells
  Processed 27_000 / 28_871 cells
  Processed 28_871 / 28_871 cells
  Writing complete in 171.74ms
Converting the cell-based data into the gene-based format.


  Counted 17_524_579 non-zeros over 12_526 genes in 6.84ms; converting in 1 phase(s).
  Phase 1/1: genes 0-12526, scatter 26.32ms, compress + write 64.09ms
  Conversion into gene-friendly format done: 98.19ms


Drop the doublets next. The `multiplets` column is the demultiplexing call,
and a doublet is two cells' counts added together, which is not something any
of these models should see. A handful of cells also have no cell type call;
they go too.

In [3]:
obs = sc.obs(cols=["cell_idx", "multiplets", "cell"], filtered=True)

keep = obs.filter((pl.col("multiplets") == "singlet") & pl.col("cell").is_not_null())[
    "cell_idx"
]

sc.set_cells_to_keep(keep.to_numpy())

sc

SingleCells(
  dir='/var/folders/l0/d50zd8812jg94mhtl8t2swfc0000gn/T/tmpwdzj5o4g/kang',
  cells=28871 (kept 24562), genes=12526,
  hvg=False, pca=False, embeddings=none,
  knn=False, graph=False, magic=False, residuals=False,
  stale=none
)

Look at the design before anything else.

In [4]:
obs = sc.obs(filtered=True)

obs.pivot(on="stim", index="ind", values="cell_id", aggregate_function="len").sort(
    "ind"
)

ind,ctrl,stim
i32,u32,u32
101,858,1166
107,517,525
1015,2738,2352
1016,1723,1635
1039,408,584
1244,1879,1464
1256,2108,2026
1488,2030,2549


Eight donors, both arms, hundreds to thousands of cells each. That is what
NEBULA needs. It also needs enough cells per donor: below thirty it quietly
downgrades from its LN method to HL, as the original package does.

## One cell type at a time

Differential expression across a mixed population mostly measures composition.
If the treatment shifts the monocyte fraction, every monocyte gene comes out
"differential" whether or not any monocyte changed. Work within a cell type.

In [5]:
mono = bv.SingleCellsSubset(sc, "cell", "CD14+ Monocytes")

bv.pp.find_hvg(mono, hvg_no=500)
hvgs = mono.gene_ids(mono.map.hvg)

mono

HVG (VST): 12_526 genes, 1 batches, 1 disk block(s)
HVG (VST): Calculated gene statistics in 13.34ms
HVG (VST): Fitted Loess in 527.92µs
HVG (VST): Standardised variance in 417.00µs (569 gene(s) re-read for the clip)
HVG (VST): Total run time -> 14.33ms


SingleCellsSubset(
  dir='/var/folders/l0/d50zd8812jg94mhtl8t2swfc0000gn/T/tmpwdzj5o4g/kang',
  group=cell = CD14+ Monocytes,
  cells=5355, genes=12526,
  hvg=True, pca=False, embeddings=none,
  knn=False, graph=False, magic=False, residuals=False,
  stale=none
)

Restricting to the highly variable genes is not just for speed. NEBULA fits an
optimiser per gene, and running it over 12,000 genes when 500 highly variable
ones will do is a poor trade.

## The real contrast

IFN-β stimulation is about as strong a perturbation as single cell data offers.
All three approaches should find it. Same gene set for all three, so the counts
are comparable.

In [6]:
mono_obs = mono.obs()

wilcox_res = bv.de.find_markers(
    mono,
    cells_1=mono_obs.filter(pl.col("stim") == "stim")["cell_id"].to_list(),
    cells_2=mono_obs.filter(pl.col("stim") == "ctrl")["cell_id"].to_list(),
    verbose=True,
)

wilcox_res.filter(pl.col("gene_id").is_in(hvgs) & (pl.col("fdr") <= 0.05)).height

Loaded in data: 4.39ms
Finished the ranking across cells: 16.74ms
Finished DGE calculations: 504.33µs


144

For pseudobulk, the samples are donor by condition, so sixteen of them. The
design blocks on the donor, which is what makes it a paired test. The design
is a plain matrix with one row per pseudobulk sample in `cell_list` order;
a polars frame keeps the column names, so the coefficient can be picked by
name.

In [7]:
groups = (
    mono_obs.with_columns(pl.col("ind").cast(pl.String), pl.col("stim").cast(pl.String))
    .group_by("ind", "stim", maintain_order=True)
    .agg("cell_id")
    .sort("ind", "stim")
)

cell_list = {
    f"{ind}_{stim}": cells
    for ind, stim, cells in groups.select("ind", "stim", "cell_id").iter_rows()
}

donor_levels = sorted(groups["ind"].unique())
design_pb = pl.DataFrame(
    {
        "(Intercept)": np.ones(groups.height),
        **{f"ind{d}": (groups["ind"] == d).cast(pl.Float64) for d in donor_levels[1:]},
        "stimstim": (groups["stim"] == "stim").cast(pl.Float64),
    }
)

pb_res = bv.de.pseudobulk_dge(
    mono,
    cell_list,
    design_pb,
    coef="stimstim",
    edger_params=EdgerQlParams(filter=False),
    verbose=True,
)

pb_res.filter(pl.col("feature_id").is_in(hvgs) & (pl.col("fdr") <= 0.05)).height

Processed group 10 out of 16 (took 367.00µs, completed 62.5%)


247

NEBULA takes the design as a formula against the obs table and the donor column
separately. The donor is not a term in the design; it is the random effect.

In [8]:
nebula_res = bv.de.nebula(
    mono,
    subject_col="ind",
    design="~ stim",
    genes_to_use=hvgs,
    verbose=True,
)

nebula_res

  Progress: 100% (500 / 500 genes, 189.62ms)
NEBULA: fitted 425 of 500 genes in 190.60ms.


ScNebula(
  n_genes=425,
  n_coefficients=2,
  method='ln',
  subject_col='ind',
  tested='stimstim',
  n_subjects=8,
  n_cells=5355,
  n_failed=0,
  n_collapsed=91,
)

In [9]:
nebula_res.results.sort("p_value", maintain_order=True).select(
    "gene_id", "log_fc", "z", "p_value", "fdr"
).head(8)

gene_id,log_fc,z,p_value,fdr
str,f64,f64,f64,f64
"""APOBEC3B""",4.666687,70.907509,0.0,0.0
"""CCL8""",5.968163,84.06467,0.0,0.0
"""CXCL10""",5.614731,108.353326,0.0,0.0
"""CTSC""",2.641504,46.266847,0.0,0.0
"""IL8""",-2.06757,-52.809536,0.0,0.0
"""CXCL11""",5.931993,44.589627,0.0,0.0
"""FAM26F""",3.230249,54.80334,0.0,0.0
"""IL1RN""",4.216952,56.464707,0.0,0.0


CXCL10, CCL8 and APOBEC3B at the top, IL8 down. That is the interferon
response, and it is reassuring rather than surprising.

Note that the Wilcoxon test finds *fewer* genes here than the pseudobulk
test (144 against 247 at FDR 0.05), not more. Pseudoreplication inflates significance, but the Wilcoxon test is also
a rank test with an expression-proportion filter, and it is simply less
powerful for a large shift in mean. On a real, strong effect all three agree
the effect is there. The difference shows up somewhere else.

### Selecting cells without a subset

You do not need a `SingleCellsSubset` to restrict the fit. `cells_to_use` takes
cell names and runs NEBULA on those cells of the full object. Handy when the
selection is a one-off, say one cell type in one condition, and building a
subset for it is overkill. Same cells, same fit:

In [10]:
nebula_res_cells = bv.de.nebula(
    sc,
    subject_col="ind",
    design="~ stim",
    genes_to_use=hvgs,
    cells_to_use=mono_obs["cell_id"].to_list(),
)

assert nebula_res.results.equals(nebula_res_cells.results)

  Progress: 100% (500 / 500 genes, 186.90ms)
NEBULA: fitted 425 of 500 genes in 187.83ms.


### Genes carried by one donor

NEBULA's expression filter, `cpc` and `mincp`, pools every cell, so a single
donor expressing a gene is enough to get it fitted. `min_subjects` adds a
per-donor check: the gene has to clear `cpc` within at least that many donors.
It defaults to `0`, which is off and matches the `nebula` package.

In [11]:
nebula_res_min = bv.de.nebula(
    mono,
    subject_col="ind",
    design="~ stim",
    genes_to_use=hvgs,
    nebula_params=NebulaParams(min_subjects=6),
)

{
    "default": nebula_res.results.height,
    "min_subjects": nebula_res_min.results.height,
}

  Progress: 100% (500 / 500 genes, 161.83ms)
NEBULA: fitted 359 of 500 genes in 162.62ms.


{'default': 425, 'min_subjects': 359}

## No contrast at all

Here is the test that matters. Take the control cells only, so there is no
treatment anywhere in the data, and split the eight donors into two arbitrary
groups of four. Any gene called differential is a false positive by
construction.

In [12]:
ctrl = mono_obs.filter(pl.col("stim") == "ctrl")
donors = sorted(ctrl["ind"].unique())
group_a = donors[0::2]

ctrl = ctrl.with_columns(
    fake=pl.when(pl.col("ind").is_in(group_a)).then(pl.lit("a")).otherwise(pl.lit("b"))
)

ctrl.pivot(on="fake", index="ind", values="cell_id", aggregate_function="len").select(
    "ind", "a", "b"
).sort("ind")

ind,a,b
i32,u32,u32
101,202,0
107,0,209
1015,779,0
1016,0,369
1039,112,0
1244,0,415
1256,390,0
1488,0,294


In [13]:
wilcox_null = bv.de.find_markers(
    mono,
    cells_1=ctrl.filter(pl.col("fake") == "a")["cell_id"].to_list(),
    cells_2=ctrl.filter(pl.col("fake") == "b")["cell_id"].to_list(),
)

wilcox_null.filter(pl.col("gene_id").is_in(hvgs) & (pl.col("fdr") <= 0.05)).height

Loaded in data: 2.71ms


Finished the ranking across cells: 7.39ms
Finished DGE calculations: 483.42µs


45

In [14]:
groups_null = ctrl.group_by("ind").agg("cell_id", pl.col("fake").first()).sort("ind")

cell_list_null = {
    str(ind): cells for ind, cells in groups_null.select("ind", "cell_id").iter_rows()
}

design_null = pl.DataFrame(
    {
        "(Intercept)": np.ones(groups_null.height),
        "fakeb": (groups_null["fake"] == "b").cast(pl.Float64),
    }
)

pb_null = bv.de.pseudobulk_dge(
    mono,
    cell_list_null,
    design_null,
    edger_params=EdgerQlParams(filter=False),
)

pb_null.filter(pl.col("feature_id").is_in(hvgs) & (pl.col("fdr") <= 0.05)).height

0

A `SingleCellsSubset` cuts a `SingleCells`, not another subset, so fold both
restrictions into one grouping column on the parent.

In [15]:
parent_obs = sc.obs(cols=["cell_idx", "cell", "stim", "ind"], filtered=True)

sc.add_obs(
    parent_obs.select(
        "cell_idx",
        cell_stim=pl.concat_str(
            pl.col("cell").cast(pl.String),
            pl.col("stim").cast(pl.String),
            separator="__",
        ),
        fake=pl.when(pl.col("ind").is_in(group_a))
        .then(pl.lit("a"))
        .otherwise(pl.lit("b")),
    )
)

mono_ctrl = bv.SingleCellsSubset(sc, "cell_stim", "CD14+ Monocytes__ctrl")

nebula_null = bv.de.nebula(
    mono_ctrl,
    subject_col="ind",
    design="~ fake",
    genes_to_use=hvgs,
    verbose=True,
)

nebula_null.results.filter(pl.col("fdr") <= 0.05).height

  Progress: 100% (500 / 500 genes, 107.86ms)
NEBULA: fitted 415 of 500 genes in 108.73ms.


6

The cell-level test calls dozens of genes differential between two groups of
donors that differ in nothing. The donor-aware tests call almost none. That gap
is the entire argument, and it does not go away with more cells: sequencing
more cells per donor makes the Wilcoxon test *more* confident about a
difference that is not there.

NEBULA is not exactly zero, and it should not be advertised as such. Eight
donors split four against four is a small experiment, real donors genuinely
differ, and an arbitrary split can land partway along some real axis of donor
variation. A handful of calls out of several hundred genes is the honest
expectation.

## Reading the rest of the output

The Wald test is the headline, but the fit carries the variance decomposition
that motivated the model.

In [16]:
nebula_res.results.sort("subject_overdispersion", descending=True).select(
    "gene_id",
    "subject_overdispersion",
    "cell_overdispersion",
    "sigma_at_bound",
    "convergence",
).head(6)

gene_id,subject_overdispersion,cell_overdispersion,sigma_at_bound,convergence
str,f64,f64,bool,i32
"""APOBEC3B""",1.749769,0.512098,false,1
"""HLA-DRB5""",1.332351,0.696977,false,1
"""GJB2""",0.980305,9.550365,false,1
"""TMEM176B""",0.945358,1.088119,false,1
"""AKR1C1_ENSG00000187134""",0.93143,64.414817,false,1
"""TMEM176A""",0.915204,2.687036,false,1


`subject_overdispersion` is how much the gene's expression moves between
donors, and it is exactly the term a cell-level test pretends is zero. Genes
with a large value are the ones a Wilcoxon test would most confidently get
wrong.

Two columns are diagnostics rather than results. `convergence` at or below
`-20` marks a fit that likely failed, and `sigma_at_bound` marks a gene where
the subject-level variance finished pinned on its lower bound, meaning the
mixed model collapsed to a plain negative binomial. Neither is fatal, but a
sweep where most genes show either is telling you something about the design.

In [17]:
nebula_res.results.select(
    n=pl.len(),
    failed=(pl.col("convergence") <= -20).sum(),
    collapsed=pl.col("sigma_at_bound").sum(),
)

n,failed,collapsed
u32,u32,u32
425,0,91


The fixed effects and their standard errors sit on the result as frames of
genes by coefficients (`coefficients` and `se`), so the full model is
available without refitting.

In [18]:
nebula_res.coefficients.head(4)

gene_id,(Intercept),stimstim
str,f64,f64
"""HBB""",-7.310557,0.510598
"""APOBEC3B""",-8.637594,4.666687
"""HBA2""",-8.767422,0.243379
"""HBA1""",-9.041931,0.071339


## Which one to reach for

Use pseudobulk when the number of donors is decent and you want something
conservative, well understood and fast. It is a bulk method with decades of
scrutiny behind it, and `bv.de.run_edger_ql()` gives you the edgeR
quasi-likelihood chain in Rust.

Use NEBULA when the cell-level information is worth keeping: few donors, uneven
cell counts, or when you care about the variance decomposition rather than only
the fold change.

Use the Wilcoxon test for marker detection, which is what it is good at. Asking
"which genes distinguish this cluster" is a description of the data in front of
you, not an inference about a population of donors, so pseudoreplication does
not apply. Asking "does this treatment change expression" is an inference, and
then it does.

## Cost

NEBULA fits an optimiser per gene and the fits are independent, so it
parallelises well and scales linearly in genes. On this data 500 genes over
5,000 cells take a fraction of a second (NEBULA prints its timing above with
`verbose=True`). Ten times the cells and ten times the genes is a hundred
times the work, which is fine for a considered analysis and painful in a loop
over twenty cell types. Restrict the genes.

`gene_batch_size` in `NebulaParams` bounds how much of the store is resident
at once. It changes nothing about the answer, since NEBULA is
gene-independent, so leave it alone unless memory is tight.

## Where next

The differential *abundance* question is the other half of this, and it has its
own [vignette](differential_abundance.ipynb): whether the treatment changed
which cells are there, rather than what those cells are expressing. The same
donor structure argument applies, and Milo handles it the same way.

## Clean up

In [19]:
tempdir_kang.cleanup()